# 🤖 LifeMate Agent - Kaggle Evaluation
### Zalo AI Challenge 2026 - Voice-based Personal Assistant Agent

Notebook này hướng dẫn đánh giá **LifeMate Agent** trên Kaggle, bao gồm cấu hình nhiều GPU khi có.

#### ⚡ Điểm nổi bật:
- Mỗi GPU worker nạp riêng một bản model; hai GPU chạy data parallel, không gộp VRAM.
- Mặc định giới hạn 512 token mới mỗi lượt. Nếu vẫn OOM, thử model 2B hoặc 256 token; giới hạn này chủ yếu giảm KV cache, không giảm đỉnh bộ nhớ prefill của prompt hiện tại.
- Bật `expandable_segments` để giảm phân mảnh CUDA memory. T4 dùng FP16; T4 (compute capability 7.5) không có hỗ trợ BF16 phần cứng và FlashAttention-2 CUDA upstream không hỗ trợ Turing.
- Mỗi lần chạy lưu `run.log`; chế độ nhiều GPU lưu thêm `worker_<gpu>.log`.
- Chấm điểm theo `state_match`, ghép cặp tool calls và Token-F1 (> 0.6) cho trường fuzzy.
- Tự nhận diện dữ liệu trong `/kaggle/input` hoặc `./data`.


## 1. Kiểm tra phần cứng GPU (Tesla T4 x 2)

In [ ]:
!nvidia-smi

import os
# Set before importing/initializing CUDA; helps with the fragmentation reported by PyTorch.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
GPU_LIST = list(range(min(device_count, 2))) if device_count else None
GPU_CLI_ARGS = f"--gpus {','.join(map(str, GPU_LIST))}" if GPU_LIST else ""
print(f"Detected GPUs: {device_count}; CUDA allocator: {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")
for i in range(device_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  - GPU {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")
if device_count < 2:
    print("⚠️ Chỉ phát hiện tối đa 1 GPU. Nếu OOM, giảm model size; dùng 1 GPU không làm giảm VRAM cần cho model trên GPU đó.")


## 2. Cài đặt các thư viện cần thiết
Cài đặt các gói hỗ trợ mô hình LLM, 4-bit NF4 Quantization (BitsAndBytes), ASR và âm thanh.

In [ ]:
!pip install -q transformers==5.21.0 accelerate bitsandbytes torchvision soundfile librosa

## 3. Thiết lập Repository & Thư mục làm việc
Đoạn mã dưới đây đảm bảo thư mục làm việc và `PYTHONPATH` được cấu hình chính xác dù bạn tải repository từ GitHub hoặc upload notebook vào Kaggle.

In [ ]:
import os
import sys
from pathlib import Path

# Kiểm tra nếu repo đã có sẵn hoặc cần clone
if Path("evaluate.py").is_file():
    REPO_DIR = Path(".").resolve()
elif Path("/kaggle/working/LifeMate").is_dir():
    REPO_DIR = Path("/kaggle/working/LifeMate").resolve()
else:
    print("Cloning repository...")
    !git clone https://github.com/NT-Loi/LifeMate.git /kaggle/working/LifeMate
    REPO_DIR = Path("/kaggle/working/LifeMate").resolve()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Thư mục làm việc: {os.getcwd()}")

## 4. Kiểm tra dữ liệu ZAC 2026
`config.py` đã được tích hợp tự động dò tìm dữ liệu trong `/kaggle/input` nếu bạn add dataset vào Kaggle, hoặc dùng thư mục `./data` có sẵn trong repo.

In [ ]:
from config import MODE, DATA_ROOT
from huggingface_hub import snapshot_download
from models.llm.qwen import MODEL_REVISIONS

MODEL_ID = "Qwen/Qwen3.5-2B"  # Change to another supported model ID to compare models.
MAX_NEW_TOKENS = 512  # Reduce to 256 if GPU memory is still insufficient.
snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISIONS[MODEL_ID])


print(f"📁 Thư mục dữ liệu: {DATA_ROOT}")
for split in ["train", "public_test"]:
    cfg = MODE[split]
    cat_exists = Path(cfg["catalog_path"]).is_file()
    data_exists = Path(cfg["data_path"]).is_file()
    tools_exists = Path(cfg["tools_dir"]).is_dir()
    print(f"[{split.upper()}] Data: {'✅' if data_exists else '❌'} | Catalog: {'✅' if cat_exists else '❌'} | zaic_tools: {'✅' if tools_exists else '❌'}")

## 5. Chạy đánh giá thử (CLI)
Mỗi GPU worker nạp riêng một model. `GPU_CLI_ARGS` tự chọn tối đa hai GPU tìm được; bắt đầu với 10 mẫu và token cap. Nếu OOM với model 4B, ưu tiên dùng model 2B; nếu OOM xuất hiện sau nhiều lượt sinh, giảm `MAX_NEW_TOKENS` xuống 256. Với model 4B cần chia qua hai GPU, có thể thử `--gpus auto` để sharding một model (chạy tuần tự, không data parallel).


In [ ]:
# Đánh giá thử 10 mẫu đầu tiên trên cả 2 GPU T4 (0 và 1)
!python evaluate.py --mode train --num-samples 10 {GPU_CLI_ARGS} --model-id $MODEL_ID --max-new-tokens $MAX_NEW_TOKENS

## 6. Xem kết quả bằng Python API hoặc đọc kết quả đã có
Cell này mặc định đọc `eval_details.jsonl` của lần chạy CLI ở trên, nên không chạy lại model. Đặt `RUN_API_EVALUATION = True` nếu muốn chạy riêng qua Python API.


In [ ]:
import json
import pandas as pd
from pathlib import Path
from config import MODE
from evaluate import run_evaluation

RUN_API_EVALUATION = False
if RUN_API_EVALUATION:
    # API path: chạy lại 10 mẫu đầu tiên.
    samples = []
    with open(MODE["train"]["data_path"], "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                samples.append(json.loads(line))
            if len(samples) >= 10:
                break
    results = run_evaluation(
        samples=samples,
        mode="train",
        gpus=GPU_LIST,
        model_id=MODEL_ID,
        max_new_tokens=MAX_NEW_TOKENS,
    )
else:
    details_path = Path("eval_results/latest/eval_details.jsonl")
    if not details_path.is_file():
        raise FileNotFoundError("Run the CLI evaluation first, or set RUN_API_EVALUATION = True.")
    with details_path.open("r", encoding="utf-8") as f:
        results = [json.loads(line) for line in f if line.strip()]

# Hiển thị bảng tổng kết DataFrame
df = pd.DataFrame(results)
cols = ["id", "state_match", "p", "r", "f1", "score", "error"]
display_cols = [c for c in cols if c in df.columns]
df_styled = df[display_cols].style.map(
    lambda v: "background-color: #d4edda; color: #155724; font-weight: bold;" if v is True else (
        "background-color: #f8d7da; color: #721c24; font-weight: bold;" if v is False else ""
    ),
    subset=["state_match"]
)
df_styled


## 7. Đánh giá chuyên sâu 1 mẫu cụ thể (Single Sample Diagnostic)
Khi cần debug chi tiết từng bước ReAct loop của một sample cụ thể (ví dụ: `train_0001`):

In [ ]:
!python evaluate.py --id train_0001 --model-id $MODEL_ID --max-new-tokens $MAX_NEW_TOKENS

## 8. Tệp kết quả và log
Mỗi lần chạy lưu vào thư mục có timestamp và cập nhật bản mới nhất trong `eval_results/latest/`:
1. `predictions.jsonl`: dự đoán theo format nộp bài.
2. `eval_details.jsonl`: điểm, calls, hash, messages và lỗi từng sample.
3. `conversations.txt`: lịch sử hội thoại dễ đọc.
4. `eval_summary.json`: chỉ số tổng kết và đường dẫn artifact.
5. `run.log`: stdout/stderr và log Python của tiến trình chính. Chạy nhiều GPU thì xem thêm `worker_<gpu>.log` để thấy log nạp model và OOM của từng worker.


In [ ]:
# Liệt kê các tệp kết quả mới nhất
!ls -lh eval_results/latest/

# Đọc thử 2 dòng đầu file dự đoán predictions.jsonl
!head -n 2 eval_results/latest/predictions.jsonl

# Đọc thử lịch sử hội thoại của sample đầu tiên trong conversations.txt
!head -n 50 eval_results/latest/conversations.txt

# Xem log tiến trình chính (bao gồm progress và traceback)
!tail -n 80 eval_results/latest/run.log

# Hiển thị báo cáo eval_summary.json
import json
if Path("eval_results/latest/eval_summary.json").is_file():
    with open("eval_results/latest/eval_summary.json", "r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

## 9. Chạy Đánh giá toàn bộ tập dữ liệu (Full Train Evaluation)
Chạy đánh giá toàn bộ các mẫu của tập train (`--num-samples -1`) trên cả 2 GPU T4.
> 💡 Bỏ comment dòng bên dưới khi bạn sẵn sàng chạy full tập train.

In [ ]:
# !python evaluate.py --mode train --num-samples -1 {GPU_CLI_ARGS} --model-id $MODEL_ID --max-new-tokens $MAX_NEW_TOKENS

## 10. Chọn mô hình
Đổi `MODEL_ID` ở ô tải model để so sánh các ID được đăng ký trong `models/llm`. Các ô đánh giá sẽ dùng cùng ID và revision đã tải vào cache.

In [ ]:
# Smoke test trên 1 GPU; lưu ý cách này không giảm peak VRAM của model.
# !python evaluate.py --num-samples 5 --gpus 0 --model-id $MODEL_ID --max-new-tokens 256

# Nếu model 4B OOM trên T4 x 2, thử sharding model qua cả hai GPU:
# !python evaluate.py --num-samples 5 --gpus auto --model-id $MODEL_ID --max-new-tokens 256
